In [ ]:
import pandas as pd
IDX_PARQUET = "metaspace_images_dump/msi_fm_samples.parquet"
idx = pd.read_parquet(IDX_PARQUET)

MAN_PARQUET  = "metaspace_images_dump/manifest_expanded.parquet"
man = pd.read_parquet(MAN_PARQUET)

In [2]:
import re
def _clean(s):
    if pd.isna(s): return None
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s

def canonicalize_labels(df):
    df = df.copy()

    # 1) Polarity
    pol_map = {"pos":"Positive","positive":"Positive","+":"Positive",
               "neg":"Negative","negative":"Negative","-":"Negative"}
    def canon_polarity(s):
        if s is None: return None
        t = _clean(s).lower()
        t2 = pol_map.get(t, t)
        if t2 in ("positive","negative"):
            return t2.capitalize()
        if "pos" in t: return "Positive"
        if "neg" in t: return "Negative"
        return _clean(s)
    if "polarity" in df.columns:
        df["polarity"] = df["polarity"].map(canon_polarity)

    # 2) Ionisation Source (map DESI-MSI -> DESI, etc.)
    def canon_ion_src(s):
        if s is None: return None
        t_raw = _clean(s)
        t = t_raw.upper().replace("-", "").replace("_","")
        if "APSMALDI" in t: return "AP-SMALDI"
        if "IRMALDESI" in t or "IRMALDI" in t: return "IR-MALDESI"
        if "APMALDI" in t: return "AP-MALDI"
        if "DESIMSI" in t: return "DESI"
        if "DESI" in t: return "DESI"
        if "MALDI" in t: return "MALDI"
        return t_raw
    if "ionisationSource" in df.columns:
        df["ionisationSource"] = df["ionisationSource"].map(canon_ion_src)

    # 3) Analyzer Type
    def canon_analyzer(s):
        if s is None: return None
        t = _clean(s); tl = t.lower()
        if "timstof" in tl and "flex" in tl: return "timsTOF Flex"
        if "fticr" in tl:
            if "12t" in tl: return "12T FTICR"
            if "7t" in tl and "scimax" in tl: return "FTICR scimaX 7T"
            return "FTICR"
        if "orbitrap" in tl or "q-exactive" in tl: return "Orbitrap"
        if "tof" in tl and "reflector" in tl: return "TOF reflector"
        if tl.strip() == "qtof": return "qTOF"
        return t
    if "analyzerType" in df.columns:
        df["analyzerType"] = df["analyzerType"].map(canon_analyzer)

    # 4) Organism
    def canon_organism(s):
        if s is None: return None
        t = _clean(s); tl = t.lower()
        if "|" in t or "," in t:
            if ("human" in tl or "homo sapiens" in tl) and ("mouse" in tl or "mus musculus" in tl):
                return "Mixed"
        if "homo sapiens" in tl or tl.strip() in {"human","h. sapiens","homo"}:
            return "Homo sapiens"
        if "mus musculus" in tl or tl.strip() in {"mouse","m. musculus"}:
            return "Mus musculus"
        return t
    if "organism" in df.columns:
        df["organism"] = df["organism"].map(canon_organism)

    # 5) Organism_Part
    def canon_part(s):
        if s is None: return None
        t = _clean(s); tl = t.lower()
        if "kidney" in tl: return "Kidney"
        if "brain"  in tl: return "Brain"
        if "liver"  in tl: return "Liver"
        if "lung"   in tl: return "Lung"
        if "breast" in tl: return "Breast"
        if "skin"   in tl: return "Skin"
        if "heart"  in tl or "cardiac" in tl: return "Heart"
        return t
    if "Organism_Part" in df.columns:
        df["Organism_Part"] = df["Organism_Part"].map(canon_part)

    # 6) Condition (keep "NA" but we'll exclude it later)
    def canon_condition(s):
        if s is None: return None
        t = _clean(s); tl = t.lower()
        if tl in {"n/a","na","none","not available",""}: return "NA"
        if tl in {"biopsy","biopsies"}: return "Biopsy"
        if "fresh frozen" in tl or "frozen" in tl: return "Frozen"
        if "tumor" in tl or "tumour" in tl: return "Tumor"
        if "cancer" in tl: return "Cancer"
        if "wildtype" in tl or tl == "wt": return "Wildtype"
        if "healthy" in tl or "control" in tl: return "Healthy"
        if "diseased" in tl or "disease" in tl: return "Diseased"
        return t
    if "Condition" in df.columns:
        df["Condition"] = df["Condition"].map(canon_condition)

    return df

In [ ]:
import os
SPLIT_CSV    = os.path.join("splits_by_dataset_id.csv")
need_cols = [
    "dataset_id", "organism", "polarity", "Organism_Part", "Condition",
    "analyzerType", "ionisationSource"
]
man_sub = man[[c for c in need_cols if c in man.columns]].drop_duplicates("dataset_id")

df_meta = idx.merge(man_sub, on="dataset_id", how="left", suffixes=("", "_man"))
df_meta = df_meta.loc[:, ~df_meta.columns.duplicated()].copy().reset_index(drop=True)
splits = pd.read_csv(SPLIT_CSV)
df_meta = df_meta.merge(splits, on="dataset_id", how="left")

# Dedup metadata by sample_path and canonicalize labels
if df_meta.duplicated("sample_path").sum():
    print("[WARN] duplicate sample_path rows in metadata; keeping first.")
    df_meta = df_meta.drop_duplicates("sample_path", keep="first").reset_index(drop=True)
df_meta = canonicalize_labels(df_meta)
df_meta.to_csv("df_meta.csv")

[WARN] duplicate sample_path rows in metadata; keeping first.


In [15]:
import pandas as pd

# --------------------------------------------
# Load metadata
# --------------------------------------------
df = pd.read_csv("df_meta.csv")

# Metadata columns to summarize
meta_cols = [
    "Condition",
    "Organism_Part",
    "analyzerType",
    "ionisationSource",
    "organism",
    "polarity",
]

# Ensure the split column exists
assert "split" in df.columns, "df_meta must include a 'split' column"

# --------------------------------------------
# Build split-by-class summary table
# Each row = a metadata label
# Columns = Train / Val / Test / Total counts
# --------------------------------------------
tables = {}

for col in meta_cols:
    pivot = (
        df.pivot_table(index=col, columns="split", values="sample_path", aggfunc="count", fill_value=0)
        .rename_axis(None, axis=1)
    )
    
    # Ensure all split columns exist even if zero
    for s in ["train", "val", "test"]:
        if s not in pivot.columns:
            pivot[s] = 0

    pivot["Total"] = pivot[["train", "val", "test"]].sum(axis=1)
    pivot = pivot[["train", "val", "test", "Total"]].sort_values("Total", ascending=False)

    tables[col] = pivot

# Print each summary to inspect
for col, tbl in tables.items():
    print("\n\n===== ", col, " =====")
    print(tbl)



=====  Condition  =====
                     train  val  test  Total
Condition                                   
Wildtype               434   41   137    612
Biopsy                 457   20   121    598
Frozen                 210   24    65    299
Tumor                  174   11    30    215
Diseased               129    4    30    163
...                    ...  ...   ...    ...
patient #7               1    0     0      1
s-2008-000608, 25um      1    0     0      1
s-2008-000608, 50um      1    0     0      1
s-2203-016117, 50um      1    0     0      1
untreated                0    0     1      1

[134 rows x 4 columns]


=====  Organism_Part  =====
                       train  val  test  Total
Organism_Part                                 
Kidney                  1296   91   345   1732
Brain                    406   52   183    641
Liver                    177    8    62    247
Lung                     153    5    55    213
Skin                     125   10    26    161
...   

In [18]:
import re

latex_output = ""

for col, tbl in tables.items():

    caption = (
        f"Train/validation/test distribution for {col}. "
        "Counts represent number of MSI tiles per class."
    )
    label = f"tab:split_{col.replace(' ', '_').lower()}"

    # --- generate LaTeX using pandas first (caption at TOP) ---
    tbl_latex = tbl.to_latex(
        index=True,
        escape=True,
        column_format="lrrrr",
        longtable=True,
        caption=caption,
        label=label,
    )

    # --- remove the TOP caption+label block ---
    tbl_latex = re.sub(
        r"\\caption\{.*?\}\s*\n\\label\{.*?\}\\\\\n",
        "",
        tbl_latex,
        count=1,
        flags=re.DOTALL,
    )

    # --- remove the \caption[]{...} inside first head (continuation header) ---
    tbl_latex = re.sub(
        r"\\caption\[\]\{.*?\} \\\\\n",
        "",
        tbl_latex,
        count=1,
        flags=re.DOTALL,
    )

    # --- insert caption at the BOTTOM just before \endlastfoot ---
    tbl_latex = tbl_latex.replace(
        "\\endlastfoot",
        f"\\caption{{{caption}}}\n\\label{{{label}}}\n\\endlastfoot",
        1
    )

    # --- wrap formatting group ---
    latex_output += (
        "% ==========================================\n"
        f"% {col} split table\n"
        "% ==========================================\n"
        "\\begingroup\n"
        "\\scriptsize\n"
        "\\setlength{\\tabcolsep}{6pt}\n"
        "\\renewcommand{\\arraystretch}{1.2}\n"
        + tbl_latex
        + "\\endgroup\n\n"
    )

with open("supplementary_split_tables.tex", "w") as f:
    f.write(latex_output)

print("Saved LaTeX tables to supplementary_split_tables.tex")

Saved LaTeX tables to supplementary_split_tables.tex
